# Multimapping correction recovers viral signal

**Utility:** ViralScan's combined host+virus reference plus EM multimapping
correction recovers viral UMIs that unique-only counting discards. In the EBV
benchmark (manuscript, *Combined-reference EM recovers EBV signal*), this raised
detected EBV UMIs **3.64×** over unique-only counting.

This notebook runs the *actual* EM allocator (`viralscan.multimapping.em_gene_abundances`,
the RSEM/kallisto-style model) on a small controlled example so you can see exactly
what the correction does. Requires `viralscan >= 2.3.0`.

> **Caveat (from the paper):** the EM estimates *one transcriptome-wide* abundance
> vector and allocates multimappers with it in every cell; it does not model
> cell-to-cell viral-abundance variation. This is fast and well-suited to sparse
> viral signal, but per-cell allocation may be preferable in heavily infected cultures.

## The setup: unique reads plus ambiguous equivalence classes

Each read maps to an *equivalence class* (EC) — the set of genes it is compatible
with. Reads in a single-gene EC are **unique** (fixed mass). Reads compatible with
several genes are **multimappers**; unique-only counting throws them away.

Toy reference: one host gene and two EBV accessions that share a genomic region
(so many reads multimap between them). Unique viral support is tiny; most viral
reads land in the shared-region EC.

In [ ]:
import numpy as np
from viralscan.multimapping import em_gene_abundances

genes = ["HOST_ACTB", "EBV_BALF5", "EBV_BXLF1"]  # gene 0 host, genes 1-2 viral
viral = [1, 2]

# Global unique-mapping UMI per gene (the fixed mass).
unique_per_gene = np.array([500.0, 3.0, 2.0])  # only 5 unique viral UMI total

# Multi-gene EC masses pooled across all cells: {tuple(gene indices): UMI}.
ec_counts = {
    (1, 2): 30.0,   # shared EBV region: 30 UMI ambiguous between the two EBV genes
    (0, 1): 20.0,   # host-virus ambiguous: 20 UMI shared by host + EBV_BALF5
}
print('unique-only viral UMI:', unique_per_gene[viral].sum())

## Run the EM allocator

One EM sweep = an E-step (allocate each EC's mass by current abundance) and an
M-step (`theta = unique + allocated`), iterated to a fixed point. `unique-weighted`
is exactly the first E-step; `em` iterates it.

In [ ]:
theta = em_gene_abundances(
    ec_counts=ec_counts,
    unique_per_gene=unique_per_gene,
    pseudocount=1.0,
    max_iter=100,
    tol=1e-6,
)

import pandas as pd
comparison = pd.DataFrame({
    'gene': genes,
    'unique_only': unique_per_gene,
    'em_corrected': np.round(theta, 2),
})
comparison

In [ ]:
uniq_viral = unique_per_gene[viral].sum()
em_viral = theta[viral].sum()
print(f'viral UMI  unique-only : {uniq_viral:.1f}')
print(f'viral UMI  EM-corrected: {em_viral:.1f}')
print(f'recovery factor        : {em_viral / uniq_viral:.2f}x')

Here the viral recovery (5 → ~36 UMI) comes almost entirely from the **intra-viral**
shared-region EC `(1,2)` (+30): reads ambiguous among a virus's *own* accessions are
allocated back within the viral genes. The **host-virus ambiguous** EC `(0,1)` adds
almost nothing to the virus (+0.1) — EM correctly routes ~19 of its 20 UMI to the
abundant host gene, so the correction does not inflate viral counts.

Two things make this matter at scale, both documented in the paper:

1. A virus like EBV is represented by **~96 near-identical accessions** in the combined
   index, so intra-viral shared-region multimappers are abundant — exactly the `(1,2)`
   effect above, multiplied across the panel.
2. The combined reference *keeps* host-virus ambiguous reads for EM to weigh, whereas a
   **two-step host-filter deletes them** before the viral pass — which is why, in the
   paper, two-step filtering recovered *fewer* EBV UMIs (3,096) than even unique-only
   counting (3,372).

Together these produced the **3.64×** EBV recovery over unique-only (12,255 vs 3,372 UMI).

## In a real run: choosing the method and the primary-call matrix

You do not call the allocator directly. Pick the correction with `--multimap-method`
and the matrix used for the *headline* viral numbers with `--multimap-primary-call`:

```bash
# Full EM correction during quantification:
viralscan -t t2g.txt -i index.idx -o out/ -s1 R1.fastq.gz -s2 R2.fastq.gz \
          --multimap-method em

# Already have a completed run? Swap the method WITHOUT redoing kb count:
viralscan rerun-multimap -o out/ --multimap-method em --cores 8
```

`--multimap-method` values: `equal`, `host-conservative` (default), `unique-weighted`, `em`.

`--multimap-primary-call` selects which matrix drives `total_umi` / `infected_cells`
in `viral_summary.tsv`: `legacy` (default; original + corrected), `unique-only`
(conservative — unique viral counts only), or `confidence`. Full-library denominators
(`umi_per_10k`, `viral_fraction`) always use the complete expression matrix.

## Summary

- Unique-only counting discards host-virus and intra-virus ambiguous reads.
- ViralScan's combined reference keeps them and allocates them with an EM model.
- Use `--multimap-method em`; use `rerun-multimap` to switch methods cheaply.
- Report conservatively with `--multimap-primary-call unique-only` when you want
  only unambiguous viral evidence in the headline numbers.